# 03 — Zepto RAG pipeline demo (graded mock baseline)

Companion walk-through for the `/support_assistant` module. The **served** application stays as
plain Python (`main.py` + `app/*.py`) because FastAPI + the Dockerfile must run a real Python app
via `uvicorn main:app`. This notebook reuses **the exact same objects** those modules use, so what
you see here is identical to what `POST /ask` returns. `MOCK_LLM` is left at its default, so every
generation step below is the deterministic, fully offline mock (no LLM, no network).


In [1]:
from pathlib import Path

from app.embeddings import embed_query
from app.graph import POLICY_KEYWORDS, handle_query
from app.llm import MOCK_LLM
from app.prompt import SYSTEM_PROMPT_TEMPLATE
from app.schema import AnswerResponse
from app.store import COLLECTION_NAME, count_chunks, query_chunks

print("MOCK_LLM in mock mode? ", MOCK_LLM)
print("ChromaDB collection   :", COLLECTION_NAME, "| embedded chunks:", count_chunks())
print("Intent-routing keywords:", POLICY_KEYWORDS)


MOCK_LLM in mock mode?  True
ChromaDB collection   : zepto_policies | embedded chunks: 8
Intent-routing keywords: ['delivery', 'return', 'refund', 'membership', 'tracking', 'cancel', 'gift card', 'support hours']


### Stage 1 — Ingestion: the 8-document corpus

One chunk per document (`doc_01.txt` … `doc_08.txt`), loaded by `ingest.py`.


In [2]:
docs = sorted(Path("docs").glob("doc_*.txt"))
for p in docs:
    t = p.read_text(encoding="utf-8").strip()
    print(f"{p.stem}  ({len(t):>4} chars)  {t[:88]}…")
print("\nTotal documents:", len(docs))


doc_01  ( 503 chars)  Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30…
doc_02  ( 559 chars)  Grocery and perishable items may be reported for a return within 24 hours of delivery if…
doc_03  ( 525 chars)  Zepto offers three account tiers: Basic (free, default tier, standard delivery fees appl…
doc_04  ( 427 chars)  Every Zepto order shows a live rider-tracking map from the moment it is packed until del…
doc_05  ( 497 chars)  Orders can be cancelled free of cost any time before the order status changes to 'Packed…
doc_06  ( 486 chars)  If an order arrives with damaged, spoiled, or missing items, customers must report it wi…
doc_07  ( 468 chars)  Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and …
doc_08  ( 333 chars)  Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given…

Total documents: 8


### Stage 2 — Embedding: all-MiniLM-L6-v2 (384-d, local, L2-normalized)

`app/embeddings.py` converts text to normalized 384-dim vectors; cosine similarity = dot product.


In [3]:
v_delivery = embed_query("What is Zepto's delivery time?")
v_gift = embed_query("Can I combine gift card balance with another gift card?")
print("query embedding dimension:", len(v_delivery))

import numpy as np

a, b = np.array(v_delivery), np.array(v_gift)
print("self-similarity (delivery vs itself) :", round(float(a @ a), 4))
print("similarity (delivery vs gift card)    :", round(float(a @ b), 4))


C:\Users\davpr\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1573.86it/s]

query embedding dimension: 384
self-similarity (delivery vs itself) : 1.0
similarity (delivery vs gift card)    : -0.0633


### Stage 3 — Retrieval (always runs for real): top-3 cosine search in ChromaDB

`app/store.query_chunks` embeds the query and returns `(id, text, cosine_distance)` triplets.
Note how each question retrieves the chunk whose *content actually matches the question*.


In [4]:
for q in [
    "What is Zepto's delivery time?",
    "Can I cancel my order after it has been packed?",
    "How much does Zepto Pass cost per month?",
]:
    print(q)
    for cid, txt, dist in query_chunks(q, n=3):
        print(f"   {cid}  distance={dist:.4f}  {txt[:72]}…")


What is Zepto's delivery time?
   doc_01  distance=0.3194  Zepto delivers grocery and household essentials to serviceable pin codes…
   doc_04  distance=0.3522  Every Zepto order shows a live rider-tracking map from the moment it is …
   doc_08  distance=0.3562  Zepto customer support is available via in-app chat 24 hours a day, 7 da…
Can I cancel my order after it has been packed?
   doc_05  distance=0.3227  Orders can be cancelled free of cost any time before the order status ch…
   doc_02  distance=0.6124  Grocery and perishable items may be reported for a return within 24 hour…
   doc_06  distance=0.6610  If an order arrives with damaged, spoiled, or missing items, customers m…
How much does Zepto Pass cost per month?
   doc_03  distance=0.2838  Zepto offers three account tiers: Basic (free, default tier, standard de…
   doc_01  distance=0.4432  Zepto delivers grocery and household essentials to serviceable pin codes…
   doc_08  distance=0.5483  Zepto customer support is available 

### Stage 4 — Intent routing: keyword heuristic (no LLM call)

`classify_intent` checks the lower-cased query against the mandated keyword list.


In [5]:
for q in ["What is Zepto's delivery time?", "What is the capital of France?"]:
    intent = "policy_question" if any(k in q.lower() for k in POLICY_KEYWORDS) else "general_question"
    print(f"{q!r:45} -> {intent}")


"What is Zepto's delivery time?"              -> policy_question
'What is the capital of France?'              -> general_question


### Stage 5 — Full LangGraph run + guaranteed Pydantic JSON output

Same `StateGraph` the FastAPI app serves: `classify_intent → (conditional edge) →
retrieve_and_answer | direct_answer → END`. Output is the validated schema
`{answer, sources, confidence}` — deterministic in mock mode.


In [6]:
for q in [
    "What is Zepto's delivery time?",
    "Can I cancel my order after it has been packed?",
    "What is the capital of France?",
]:
    st = handle_query(q)
    resp = AnswerResponse(answer=st["answer"], sources=st["sources"], confidence=st["confidence"])
    print(f"query  : {q}")
    print(f"intent : {st['intent']}")
    print(f"JSON   : {resp.model_dump_json()}\n")


query  : What is Zepto's delivery time?
intent : policy_question
JSON   : {"answer":"Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del","sources":["doc_01","doc_04","doc_08"],"confidence":1.0}

query  : Can I cancel my order after it has been packed?
intent : policy_question
JSON   : {"answer":"Based on the retrieved context: Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be","sources":["doc_05","doc_02","doc_06"],"confidence":1.0}

query  : What is the capital of France?
intent : general_question
JSON   : {"answer":"I can only answer questions about Zepto policies right now.","sources":[],"confidence":1.0}



### The structured prompt template (used only by the optional MOCK_LLM=0 extension)

`app/prompt.py` — role / context / task / format / length + negative constraint + few-shot example:


In [7]:
print(SYSTEM_PROMPT_TEMPLATE)


ROLE
You are ZeptoCare, Zepto's customer-support assistant for its delivery, returns,
membership, tracking, cancellation, damages, gift-card and support-hours policies.
You answer only about Zepto policies; never about other companies or general knowledge.

CONTEXT
The passages below were retrieved from Zepto's official policy corpus for this
question. Use ONLY these passages as the source of your answer:

{context}

TASK
Answer the user question based strictly on the CONTEXT above:
{question}

FORMAT
Respond with a single JSON object only — no prose, no markdown fences — with exactly
these keys:
{{"answer": string, "sources": [string], "confidence": number(0..1)}}
"sources" must list the chunk/document ids you actually used (e.g. ["doc_01"]);
use [] if none. "confidence" is a float between 0 and 1 reflecting how well the
context supports the answer.

LENGTH
Keep "answer" concise: 2-4 sentences, under 80 words.

NEGATIVE CONSTRAINT
Do not answer using information not present in the pro

**Summary:** ingestion (`ingest.py`) → embedding (`app/embeddings.py`) → indexing/retrieval
(`app/store.py` → ChromaDB `zepto_policies`) → routing + generation (`app/graph.py`, LangGraph) →
validated output (`app/schema.py`); the same objects are served over HTTP by `main.py`. Only the
generation step inside each node branches on `MOCK_LLM` (mock = graded baseline; `MOCK_LLM=0` =
optional real-LLM extension). See `README.md` for the full architecture description and the
FastAPI call transcripts.
